# 🏠 House Price Prediction — Linear Regression
### Morocco Real Estate Dataset

**Objectif :** Prédire le prix d'un bien immobilier (en MAD) à partir de ses caractéristiques.  
**Méthode :** Régression Linéaire — OLS (Ordinary Least Squares)

---
**Dataset :** 500 biens immobiliers dans 5 villes marocaines  
**Features :** Surface, chambres, age, étage, distance centre-ville, garage, jardin, ville  
**Target :** Prix en Dirhams (MAD)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.family'] = 'monospace'
sns.set_theme(style='whitegrid', palette='muted')
print("✅ Libraries imported successfully")


## 1. 📊 Exploration des données (EDA)

In [ ]:
df = pd.read_csv('../data/morocco_housing.csv')
print(f"Shape: {df.shape}")
print(f"\nTypes:\n{df.dtypes}")
df.head(10)


In [ ]:
# Statistiques descriptives
df.describe().round(2)


In [ ]:
# Distribution des prix par ville
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot prix par ville
df.boxplot(column='price_mad', by='city', ax=axes[0])
axes[0].set_title('Prix par ville (MAD)')
axes[0].set_xlabel('Ville')
axes[0].set_ylabel('Prix (MAD)')
plt.sca(axes[0])
plt.xticks(rotation=20)

# Distribution générale des prix
axes[1].hist(df['price_mad'] / 1e6, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_title('Distribution des prix')
axes[1].set_xlabel('Prix (Millions MAD)')
axes[1].set_ylabel('Fréquence')

plt.suptitle('')
plt.tight_layout()
plt.savefig('../app/static/eda_prices.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Plot saved")


In [ ]:
# Correlation heatmap
numeric_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(10, 7))
mask = np.triu(np.ones_like(numeric_df.corr(), dtype=bool))
sns.heatmap(numeric_df.corr(), mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5)
plt.title('Matrice de Corrélation', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../app/static/correlation.png', dpi=120, bbox_inches='tight')
plt.show()


## 2. 🔧 Feature Engineering & Preprocessing

In [ ]:
# One-Hot Encoding pour la variable 'city'
df_model = pd.get_dummies(df, columns=['city'], drop_first=False)

X = df_model.drop('price_mad', axis=1)
y = df_model['price_mad']

print("Features utilisées:")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2d}. {col}")
print(f"\nTarget: price_mad")
print(f"\nX shape: {X.shape}")


In [ ]:
# Train/Test split + Standardisation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)  # IMPORTANT: transform only, not fit!

print(f"Train size : {X_train.shape[0]} samples ({len(X_train)/len(X)*100:.0f}%)")
print(f"Test size  : {X_test.shape[0]} samples ({len(X_test)/len(X)*100:.0f}%)")


## 3. 📐 Théorie — Régression Linéaire OLS

La régression linéaire modélise la relation :

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + ... + \beta_n x_n$$

**L'objectif OLS** est de minimiser la **somme des carrés des résidus (SSR)** :

$$SSR = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 = \|y - X\beta\|^2$$

**Solution analytique (forme matricielle)** :

$$\hat{\beta} = (X^T X)^{-1} X^T y$$

C'est la solution exacte — pas d'itération nécessaire !


In [ ]:
# ============================================================
# IMPLÉMENTATION FROM SCRATCH (NumPy uniquement)
# ============================================================
class LinearRegressionOLS:
    def __init__(self):
        self.beta = None
        self.intercept_ = None
        self.coef_ = None
    
    def fit(self, X, y):
        # Ajouter colonne de 1 pour le biais (intercept)
        X_b = np.column_stack([np.ones(len(X)), X])
        
        # Formule normale : β = (X^T X)^-1 X^T y
        self.beta = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y
        
        self.intercept_ = self.beta[0]
        self.coef_ = self.beta[1:]
        return self
    
    def predict(self, X):
        X_b = np.column_stack([np.ones(len(X)), X])
        return X_b @ self.beta

# Entraîner le modèle from scratch
model_scratch = LinearRegressionOLS()
model_scratch.fit(X_train_scaled, y_train.values)
y_pred_scratch = model_scratch.predict(X_test_scaled)

r2_scratch = r2_score(y_test, y_pred_scratch)
rmse_scratch = np.sqrt(mean_squared_error(y_test, y_pred_scratch))
print("=== Modèle From Scratch (OLS) ===")
print(f"R²   : {r2_scratch:.4f}")
print(f"RMSE : {rmse_scratch:,.0f} MAD")


In [ ]:
# Vérification avec scikit-learn
model_sklearn = LinearRegression()
model_sklearn.fit(X_train_scaled, y_train)
y_pred_sklearn = model_sklearn.predict(X_test_scaled)

r2_sk = r2_score(y_test, y_pred_sklearn)
rmse_sk = np.sqrt(mean_squared_error(y_test, y_pred_sklearn))

print("=== Scikit-learn LinearRegression ===")
print(f"R²   : {r2_sk:.4f}")
print(f"RMSE : {rmse_sk:,.0f} MAD")
print(f"\n✅ Les deux modèles donnent des résultats identiques : {np.allclose(y_pred_scratch, y_pred_sklearn)}")


## 4. 📈 Évaluation du Modèle

In [ ]:
# Métriques complètes
mae  = mean_absolute_error(y_test, y_pred_sklearn)
mse  = mean_squared_error(y_test, y_pred_sklearn)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred_sklearn)
mape = np.mean(np.abs((y_test - y_pred_sklearn) / y_test)) * 100

print("╔══════════════════════════════════════╗")
print("║         MÉTRIQUES D'ÉVALUATION       ║")
print("╠══════════════════════════════════════╣")
print(f"║  R²   (coeff. détermination) : {r2:.4f} ║")
print(f"║  MAE  (erreur absolue moy.)  : {mae/1e3:6.1f}k ║")
print(f"║  RMSE (racine erreur quad.)  : {rmse/1e3:6.1f}k ║")
print(f"║  MAPE (erreur % absolue)     : {mape:6.2f}% ║")
print("╚══════════════════════════════════════╝")


In [ ]:
# Visualisation: Prédictions vs Réalité + Résidus
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: Réel vs Prédit
axes[0].scatter(y_test/1e6, y_pred_sklearn/1e6, alpha=0.5, color='steelblue', s=40)
lim = [y_test.min()/1e6, y_test.max()/1e6]
axes[0].plot(lim, lim, 'r--', lw=2, label='Prédiction parfaite')
axes[0].set_xlabel('Prix réel (M MAD)')
axes[0].set_ylabel('Prix prédit (M MAD)')
axes[0].set_title(f'Réel vs Prédit  (R² = {r2:.3f})')
axes[0].legend()

# Résidus
residuals = y_test - y_pred_sklearn
axes[1].scatter(y_pred_sklearn/1e6, residuals/1e3, alpha=0.5, color='coral', s=40)
axes[1].axhline(0, color='black', lw=1.5, linestyle='--')
axes[1].set_xlabel('Prix prédit (M MAD)')
axes[1].set_ylabel('Résidu (k MAD)')
axes[1].set_title('Analyse des Résidus')

plt.tight_layout()
plt.savefig('../app/static/evaluation.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Importance des features (coefficients)
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model_sklearn.coef_
}).sort_values('Coefficient', key=abs, ascending=True)

plt.figure(figsize=(10, 6))
colors = ['#e74c3c' if c < 0 else '#2ecc71' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'] / 1e3, color=colors, edgecolor='white')
plt.axvline(0, color='black', lw=1)
plt.xlabel('Coefficient (k MAD)')
plt.title('Impact de chaque feature sur le prix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../app/static/coefficients.png', dpi=120, bbox_inches='tight')
plt.show()

print("\n🟢 Coefficients positifs → augmentent le prix")
print("🔴 Coefficients négatifs → diminuent le prix")


## 5. 🔍 Hypothèses de la Régression Linéaire

Pour que les estimateurs OLS soient **BLUE** (Best Linear Unbiased Estimators), il faut vérifier :

| Hypothèse | Description |
|-----------|-------------|
| **H1** - Linéarité | La relation X→y est linéaire |
| **H2** - Homoscédasticité | Variance constante des résidus |
| **H3** - Normalité | Résidus ~ N(0, σ²) |
| **H4** - Indépendance | Pas d'autocorrélation des résidus |
| **H5** - Pas de multicolinéarité | Features non corrélées entre elles |


In [ ]:
from scipy import stats

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# QQ-Plot (normalité des résidus)
stats.probplot(residuals, dist='norm', plot=axes[0])
axes[0].set_title('Q-Q Plot — Normalité des résidus')

# Homoscédasticité
axes[1].scatter(y_pred_sklearn/1e6, np.abs(residuals)/1e3, alpha=0.5, color='purple', s=35)
axes[1].set_xlabel('Prix prédit (M MAD)')
axes[1].set_ylabel('|Résidu| (k MAD)')
axes[1].set_title('Homoscédasticité des résidus')

plt.tight_layout()
plt.show()

# Test de Shapiro-Wilk (normalité)
_, p_shapiro = stats.shapiro(residuals[:50])  # sample pour la vitesse
print(f"Test Shapiro-Wilk (p-value) : {p_shapiro:.4f}")
print("→ p > 0.05 : résidus normalement distribués ✅" if p_shapiro > 0.05 else "→ p ≤ 0.05 : légère déviation de la normalité ⚠️")


## 6. 🔮 Prédiction sur un nouveau bien

In [ ]:
import pickle

# Charger le modèle sauvegardé
with open('../models/model.pkl', 'rb') as f:
    model_saved = pickle.load(f)
with open('../models/scaler.pkl', 'rb') as f:
    scaler_saved = pickle.load(f)
with open('../models/feature_names.pkl', 'rb') as f:
    feature_names = pickle.load(f)

# Nouveau bien à Rabat
nouveau_bien = {
    'area_m2': 120,
    'rooms': 3,
    'age_years': 5,
    'floor': 4,
    'distance_center_km': 3.5,
    'has_garage': 1,
    'has_garden': 0,
    'city_Casablanca': 0,
    'city_Fes': 0,
    'city_Marrakech': 0,
    'city_Rabat': 1,
    'city_Tanger': 0,
}

df_input = pd.DataFrame([nouveau_bien])[feature_names]
X_input_scaled = scaler_saved.transform(df_input)
prediction = model_saved.predict(X_input_scaled)[0]

print("╔══════════════════════════════════════════╗")
print("║          NOUVEAU BIEN À PRÉDIRE          ║")
print("╠══════════════════════════════════════════╣")
print(f"║  Ville       : Rabat                     ║")
print(f"║  Surface     : 120 m²                    ║")
print(f"║  Chambres    : 3                         ║")
print(f"║  Âge         : 5 ans                     ║")
print(f"║  Étage       : 4                         ║")
print(f"║  Distance    : 3.5 km du centre          ║")
print(f"║  Garage      : Oui                       ║")
print("╠══════════════════════════════════════════╣")
print(f"║  💰 PRIX ESTIMÉ : {prediction:>10,.0f} MAD     ║")
print("╚══════════════════════════════════════════╝")


## ✅ Conclusion

| Aspect | Résultat |
|--------|----------|
| **Modèle** | Régression Linéaire OLS |
| **R²** | ~0.97 — 97% de la variance expliquée |
| **RMSE** | ~200k MAD d'erreur moyenne |
| **Feature la plus impactante** | Surface (m²) × ville |

### 📚 Ce qu'on a appris
- La **formule normale** β̂ = (XᵀX)⁻¹Xᵀy — solution analytique exacte
- L'importance du **preprocessing** (StandardScaler, One-Hot Encoding)
- L'**analyse des résidus** pour valider les hypothèses OLS
- L'**interprétation des coefficients** pour comprendre l'impact de chaque variable
